# HotpotQA — `chunk_cooccur_query` 检索评测 + 权重扫描

本 notebook 加载 `hotpotqa_latest_framework_index.ipynb` 生成的索引文件，
在 HotpotQA distractor dev 集上评测 **chunk 级共现查询方法 `chunk_cooccur_query`**，
指标为 **Recall@10**（同时给出 per-query 平均口径与 micro 口径）与 **MRR@10**。

并支持对 `chunk_cooccur_query` 的两个核心权重做**批量扫描对比**：

```
final_score(chunk) = BaseEvidence(chunk) × (1 + λ × PairBoost(chunk))
PairBoost          = mean_{top-K pair}( global_edge_weight × local_evidence )
```

| 权重 | 默认 | 含义 |
|---|---|---|
| `lambda_boost` (λ) | 0.3 | 共现加成在最终分里的强度；λ=0 退化为纯 BaseEvidence |
| `top_k_pair_boosts` | 3 | 每个 chunk 取多少个最强 pair 求平均 |

数据集构建与评测逻辑参考 `archive/hotpotqa_legacy/HotpotQA.ipynb`，但只评测 `chunk_cooccur_query`。

## 一、环境与路径设置

In [1]:
from pathlib import Path
import os
import sys
import time

# 定位项目根目录，并切换工作目录、加入 Python 搜索路径
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # 优先使用 CUDA
print(f"Working directory: {REPO_ROOT}")
print(f"Device: {DEVICE}")

Working directory: /home/xiaoyue/LiteSemRAG
Device: cuda


## 二、评测参数

- `INDEX_PKL_PATH`：要加载的索引文件，需与下面 `NUM_SAMPLES` 对应（索引构建时用的样本数）。
- `NUM_SAMPLES`：必须与构建该索引时的 `NUM_SAMPLES` 一致，否则 `documents` / `samples` 与索引内容不匹配。
- `TOP_K`：检索截断深度（Recall@K / MRR@K 的 K）。

In [2]:
# --- 数据集 ---
NUM_SAMPLES = 500  # 必须与构建索引时一致
HOTPOT_FILE_CANDIDATES = [
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]

# --- 索引文件 ---
# 命名规则与 hotpotqa_latest_framework_index.ipynb 保持一致：
#   litesemrag_hotpotqa_{NUM_SAMPLES}[_{INDEX_RUN_TAG}].pkl
INDEX_OUTPUT_DIR = REPO_ROOT / "cache" / "hotpotqa_latest_framework_index"
INDEX_RUN_TAG = "local"  # 缓存版本标签，需与构建索引时一致（本地 LLM 用 "local"、API 用 "api"）；None 或 "" 则不加标签
INDEX_BASENAME = f"litesemrag_hotpotqa_{NUM_SAMPLES}"
if INDEX_RUN_TAG:  # 追加版本标签，匹配不同 LLM 后端/配置生成的缓存
    INDEX_BASENAME = f"{INDEX_BASENAME}_{INDEX_RUN_TAG}"
INDEX_PKL_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}.pkl"

# --- 检索 / 评测 ---
TOP_K = 10  # Recall@K / MRR@K 的 K
EVAL_NUM_SAMPLES = None  # 评测样本数；None 表示用全部 samples
PRINT_IMPORTANT_TOKENS = False  # chunk_cooccur_query 是否打印调试 token 信息

print(f"Index pickle path: {INDEX_PKL_PATH}")
print(f"Index exists: {INDEX_PKL_PATH.exists()}")
assert INDEX_PKL_PATH.exists(), f"找不到索引文件: {INDEX_PKL_PATH}"

Index pickle path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_500_local.pkl
Index exists: True


## 三、加载索引

In [3]:
import RAG_graph
from utils import build_hotpot_retrieval_dataset, mrr_for_one_query_titles

load_start = time.time()
graph_database = RAG_graph.LiteSemRAG.load_data(str(INDEX_PKL_PATH))
print(f"加载索引耗时: {time.time() - load_start:.2f}s")

print("Graph stats:")
print(f"  docs: {len(graph_database.doc_nodes)}")
print(f"  chunks: {len(graph_database.chunk_nodes)}")
print(f"  tokens: {len(graph_database.token_nodes)}")
print(f"  phrase tokens: {len(graph_database.phrase_token_nodes)}")
print(f"  sem nodes: {len(graph_database.sem_nodes)}")
qdb = graph_database.query_database
print(f"  query database shape: {None if qdb is None else tuple(qdb.shape)}")

Loading text encoder models in device: GPU


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


加载索引耗时: 29.10s
Graph stats:
  docs: 4937
  chunks: 4937
  tokens: 59105
  phrase tokens: 41632
  sem nodes: 59221
  query database shape: (59221, 1024)


## 四、构建评测数据集

`build_hotpot_retrieval_dataset` 会优先读取 `./hotpot_QA/` 下的缓存，确保与构建索引时使用的是同一份 `documents` / `samples`。

In [4]:
file_path = next(
    (str(path) for path in HOTPOT_FILE_CANDIDATES if path.exists()),
    str(HOTPOT_FILE_CANDIDATES[0]),
)
print(f"Using HotpotQA file: {file_path}")

documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=NUM_SAMPLES)
print(f"Documents: {len(documents)}")
print(f"Samples: {len(samples)}")

eval_samples = samples if EVAL_NUM_SAMPLES is None else samples[:EVAL_NUM_SAMPLES]
print(f"Eval samples: {len(eval_samples)}")

# 预查每题的 gold 标题集合（去重小写），评测循环里复用
gold_sets = []
gold_titles_list = []
for sample in eval_samples:
    titles = [documents[i]["title"] for i in sample["gold_doc_ids"]]
    gold_titles_list.append(titles)
    gold_sets.append({t.strip().lower() for t in titles if t and t.strip()})

# 预览一条样本
print("\nQuestion:", eval_samples[0]["question"])
print("Gold titles:", gold_titles_list[0])

Using HotpotQA file: /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json
Loading cached dataset...
Loaded 4937 documents
Loaded 500 samples
Documents: 4937
Samples: 500
Eval samples: 500

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold titles: ['Scott Derrickson', 'Ed Wood']


## 五、评测函数

对每个问题调用 `chunk_cooccur_query`，把检索回的 chunk 映射成所属文档标题（**去重保序**），
与该题的 gold 标题对比，计算：

- **Recall@K (per-query)**：每题 `命中 gold 标题数 / gold 标题总数`，再对所有题取平均。
- **Recall@K (micro)**：`所有题命中 gold 标题总数 / 所有题 gold 标题总数`（旧 notebook 口径）。
- **MRR@K**：第一个命中 gold 的检索名次的倒数，再取平均。

> 注：Recall@K 基于「检索到的去重文档标题」计算，而非 chunk 数量；HotpotQA 每题通常有 2 个 gold 文档。

In [5]:
def retrieved_titles_for_question(question, lambda_boost, top_k_pair_boosts):
    """返回检索到的文档标题列表（按检索名次去重保序）。"""
    _, retrieved_chunk_ids, _ = graph_database.chunk_cooccur_query(
        question,
        top_k_chunk=TOP_K,
        top_k_pair_boosts=top_k_pair_boosts,
        lambda_boost=lambda_boost,
        print_important_tokens=PRINT_IMPORTANT_TOKENS,
    )
    titles = []
    seen = set()
    for cid in retrieved_chunk_ids:
        title = graph_database.chunk_nodes[cid].doc_node.doc_name
        if title not in seen:
            seen.add(title)
            titles.append(title)
    return titles


def evaluate_config(lambda_boost, top_k_pair_boosts):
    """在全部 eval_samples 上跑一组权重，返回指标 dict。"""
    per_query_recall_sum = 0.0  # per-query recall 累加
    total_gold_hits = 0         # micro 分子：命中 gold 标题总数
    total_gold = 0              # micro 分母：gold 标题总数
    mrr_sum = 0.0
    counted = 0

    start = time.time()
    for sample, gold_set, gold_titles in zip(eval_samples, gold_sets, gold_titles_list):
        if not gold_set:
            continue
        counted += 1

        ranked_titles = retrieved_titles_for_question(
            sample["question"], lambda_boost, top_k_pair_boosts
        )
        ranked_top_k = ranked_titles[:TOP_K]
        retrieved_set = {t.strip().lower() for t in ranked_top_k}

        num_hit = len(gold_set & retrieved_set)
        per_query_recall_sum += num_hit / len(gold_set)
        total_gold_hits += num_hit
        total_gold += len(gold_set)
        mrr_sum += mrr_for_one_query_titles(ranked_top_k, gold_titles, k=TOP_K)

    elapsed = time.time() - start
    return {
        "lambda_boost": lambda_boost,
        "top_k_pair_boosts": top_k_pair_boosts,
        "recall@%d_per_query" % TOP_K: per_query_recall_sum / counted,
        "recall@%d_micro" % TOP_K: total_gold_hits / total_gold,
        "mrr@%d" % TOP_K: mrr_sum / counted,
        "n": counted,
        "sec": elapsed,
    }

## 六、权重扫描配置

在 `WEIGHT_CONFIGS` 里列出要对比的 `(lambda_boost, top_k_pair_boosts)` 组合，下面会逐个在全部样本上评测。

- `lambda_boost = 0` 时共现加成完全关闭，可作为「纯 BaseEvidence」基线。
- 默认值为 `lambda_boost=0.3`、`top_k_pair_boosts=3`（`RAG_graph.COOCCUR_LAMBDA` / `COOCCUR_TOP_K_PAIR_BOOSTS`）。

In [6]:
# 每个元素 = (lambda_boost, top_k_pair_boosts)
WEIGHT_CONFIGS = [
    (0.0, 3),   # 基线：关闭共现加成
    (0.1, 3),
    (0.3, 3),   # 当前默认
    (0.5, 3),
    (1.0, 3),
    (0.3, 1),
    (0.3, 5),
    (0.3, 10),
]

print(f"待扫描配置数: {len(WEIGHT_CONFIGS)}，每个跑 {len(eval_samples)} 题")

待扫描配置数: 8，每个跑 500 题


## 七、运行扫描并汇总

> 注意：每个配置都会重跑全部 query（含查询解析与共现建图），耗时 ≈ 配置数 × 单配置耗时。

In [7]:
import pandas as pd

results = []
sweep_start = time.time()
for idx, (lam, topk) in enumerate(WEIGHT_CONFIGS):
    metrics = evaluate_config(lam, topk)
    results.append(metrics)
    print(
        f"[{idx + 1}/{len(WEIGHT_CONFIGS)}] "
        f"λ={lam}, top_k_pair={topk} | "
        f"R@{TOP_K}(pq)={metrics['recall@%d_per_query' % TOP_K]:.4f} "
        f"R@{TOP_K}(micro)={metrics['recall@%d_micro' % TOP_K]:.4f} "
        f"MRR@{TOP_K}={metrics['mrr@%d' % TOP_K]:.4f} "
        f"({metrics['sec']:.1f}s)"
    )

print(f"\n扫描总耗时: {time.time() - sweep_start:.1f}s")

results_df = pd.DataFrame(results)
# 按 per-query recall 排序，最优配置在最前
results_df = results_df.sort_values(
    f"recall@{TOP_K}_per_query", ascending=False
).reset_index(drop=True)
results_df

[1/8] λ=0.0, top_k_pair=3 | R@10(pq)=0.7170 R@10(micro)=0.7170 MRR@10=0.6843 (48.4s)
[2/8] λ=0.1, top_k_pair=3 | R@10(pq)=0.7160 R@10(micro)=0.7160 MRR@10=0.6820 (45.8s)
[3/8] λ=0.3, top_k_pair=3 | R@10(pq)=0.7170 R@10(micro)=0.7170 MRR@10=0.6798 (49.1s)
[4/8] λ=0.5, top_k_pair=3 | R@10(pq)=0.7150 R@10(micro)=0.7150 MRR@10=0.6769 (46.1s)
[5/8] λ=1.0, top_k_pair=3 | R@10(pq)=0.7100 R@10(micro)=0.7100 MRR@10=0.6760 (45.9s)
[6/8] λ=0.3, top_k_pair=1 | R@10(pq)=0.7150 R@10(micro)=0.7150 MRR@10=0.6803 (48.9s)
[7/8] λ=0.3, top_k_pair=5 | R@10(pq)=0.7170 R@10(micro)=0.7170 MRR@10=0.6791 (45.7s)
[8/8] λ=0.3, top_k_pair=10 | R@10(pq)=0.7160 R@10(micro)=0.7160 MRR@10=0.6792 (44.8s)

扫描总耗时: 374.8s


,lambda_boost,top_k_pair_boosts,recall@10_per_query,recall@10_micro,mrr@10,n,sec
0,0.0,3,0.717,0.717,0.684268,500,48.367700
1,0.3,3,0.717,0.717,0.679849,500,49.115356
2,0.3,5,0.717,0.717,0.679118,500,45.653486
3,0.1,3,0.716,0.716,0.682021,500,45.838684
4,0.3,10,0.716,0.716,0.679171,500,44.819037
5,0.5,3,0.715,0.715,0.676904,500,46.119265
6,0.3,1,0.715,0.715,0.680318,500,48.944418
7,1.0,3,0.710,0.710,0.676048,500,45.902708


## 八、单样本检查（可选）

用于人工核对某个问题在指定权重下的检索结果与 gold 文档。

In [8]:
inspect_index = 0
inspect_lambda = 0.3
inspect_top_k_pair = 3

sample = eval_samples[inspect_index]
print("Question:", sample["question"])
print("Answer:", sample["answer"])
print("Gold titles:", gold_titles_list[inspect_index])

retrieved_chunk, retrieved_chunk_ids, top_chunks = graph_database.chunk_cooccur_query(
    sample["question"],
    top_k_chunk=TOP_K,
    top_k_pair_boosts=inspect_top_k_pair,
    lambda_boost=inspect_lambda,
    print_important_tokens=True,
)

print("\n--- 检索到的 chunk（按名次）---")
for rank, (cid, text) in enumerate(zip(retrieved_chunk_ids, retrieved_chunk)):
    title = graph_database.chunk_nodes[cid].doc_node.doc_name
    print(f"[{rank}] chunk_id={cid} | doc={title}")
    print(f"     {text[:200]}")

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Gold titles: ['Scott Derrickson', 'Ed Wood']
important ents: ['scott derrickson', 'ed wood']
important phrases: []
important tokens: ['nationality']
query tokens: ['scott derrickson', 'ed wood', 'nationality']
low level tokens: ['scott derrickson(exact matched)', 'ed wood(exact matched)', 'nationality(exact matched)']
high level tokens: ['ed wood sr.(partial matched)', "traveller's nationality(partial matched)"]
{1: ['Token:scott derrickson,Score:10.7276'], 3: ['Token:scott derrickson,Score:7.7261'], 5: ['Token:scott derrickson,Score:9.8750'], 7: ['Token:scott derrickson,Score:12.0253'], 9: ['Token:scott derrickson,Score:8.8864'], 0: ['Token:ed wood,Score:14.9761'], 8: ['Token:ed wood,Score:13.3849'], 321: ['Token:nationality,Score:6.4177'], 971: ['Token:nationality,Score:10.2582', "Token:traveller's nationality,Score:2.6618"], 1772: ['Token:nationality,Score:10.6986'], 2248: ['Token:nationality,Score:5.33